# Notebook 1: Acoustic Feature Extraction + Model Training

**Module 2 — PronounceAI: Acoustic Feature Analysis**

This notebook covers:
1. Loading audio from LibriSpeech
2. Extracting acoustic features: MFCCs, Pitch (F0), Energy (RMS), Mel Spectrogram
3. Building a labeled dataset (good pronunciation vs. degraded/augmented audio)
4. Training a **Random Forest classifier** and a **small PyTorch neural network**
5. Saving trained models to `model/module_2/models/`

---
### Strategy
LibriSpeech contains only native, clean English speech ("good" pronunciation).
To simulate "poor" pronunciation we apply audio degradations:
- Pitch shifting (unnatural intonation)
- Adding white noise (unclear articulation)
- Time stretching (abnormal speaking rate)

This gives us a binary classification task: `1 = good`, `0 = degraded`.

## Step 1: Install Dependencies

In [ ]:
!pip install librosa numpy pandas matplotlib scikit-learn torch torchaudio soundfile joblib

## Step 2: Imports and Configuration

In [ ]:
import os
import tarfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import joblib
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR     = Path("../../../")                  # NLP_project root
LIBRI_DIR    = BASE_DIR / "librispeech" / "LibriSpeech" / "train-clean-100"
MODELS_DIR   = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE   = 16_000
MAX_SAMPLES   = 300    # samples per class; increase for better accuracy
N_MFCC        = 40     # number of MFCC coefficients

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"LibriSpeech path: {LIBRI_DIR.resolve()}")

## Step 3: Collect Audio File Paths from LibriSpeech

In [ ]:
def collect_audio_files(root: Path, ext: str = ".flac", limit: int = None) -> list:
    """Recursively collect audio file paths up to `limit`."""
    files = sorted(root.rglob(f"*{ext}"))
    if limit:
        files = files[:limit]
    return [str(f) for f in files]

if not LIBRI_DIR.exists():
    raise FileNotFoundError(
        f"LibriSpeech not found at {LIBRI_DIR}.\n"
        "Please run Notebook 1 of Module 1 first to extract the archives."
    )

audio_files = collect_audio_files(LIBRI_DIR, limit=MAX_SAMPLES)
print(f"Collected {len(audio_files)} audio files.")

## Step 4: Feature Extraction Functions

We extract four types of features per audio clip:

| Feature | Description | Shape |
|---|---|---|
| **MFCC** | Captures timbral texture (pronunciation shape) | 40 means + 40 stds = 80 |
| **Pitch (F0)** | Fundamental frequency — intonation & tone | 2 (mean, std) |
| **Energy (RMS)** | Loudness over time — clarity & stress | 2 (mean, std) |
| **Spectral Centroid** | Brightness of sound | 2 (mean, std) |

Final feature vector: **86 dimensions**

In [ ]:
def extract_features(audio_path: str, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Load an audio file and return a 1-D feature vector:
    [mfcc_means(40), mfcc_stds(40), pitch_mean, pitch_std,
     rms_mean, rms_std, centroid_mean, centroid_std]
    Returns None if loading fails.
    """
    try:
        y, _ = librosa.load(audio_path, sr=sr, mono=True)
    except Exception:
        return None

    if len(y) < sr * 0.5:   # skip clips shorter than 0.5 seconds
        return None

    # ── MFCCs ──────────────────────────────────────────────────────────────
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc_mean = np.mean(mfcc, axis=1)   # (N_MFCC,)
    mfcc_std  = np.std(mfcc,  axis=1)   # (N_MFCC,)

    # ── Pitch (F0) via librosa pyin ─────────────────────────────────────────
    f0, voiced_flag, _ = librosa.pyin(
        y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
    )
    f0_voiced = f0[voiced_flag]          # only voiced frames
    pitch_mean = np.mean(f0_voiced) if len(f0_voiced) > 0 else 0.0
    pitch_std  = np.std(f0_voiced)  if len(f0_voiced) > 0 else 0.0

    # ── Energy (RMS) ────────────────────────────────────────────────────────
    rms = librosa.feature.rms(y=y)[0]
    rms_mean = np.mean(rms)
    rms_std  = np.std(rms)

    # ── Spectral Centroid ────────────────────────────────────────────────────
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    centroid_mean = np.mean(centroid)
    centroid_std  = np.std(centroid)

    return np.concatenate([
        mfcc_mean, mfcc_std,
        [pitch_mean, pitch_std, rms_mean, rms_std, centroid_mean, centroid_std]
    ])

# Quick sanity check
sample_vec = extract_features(audio_files[0])
print(f"Feature vector dimension: {sample_vec.shape[0]}")

## Step 5: Audio Augmentation (Simulates Poor Pronunciation)

In [ ]:
def augment_audio(y: np.ndarray, sr: int = SAMPLE_RATE, mode: str = "random") -> np.ndarray:
    """
    Apply one of three degradations to simulate poor pronunciation:
      - 'pitch'   : unnatural pitch shift
      - 'noise'   : add white noise (unclear articulation)
      - 'stretch' : time stretch (abnormal speech rate)
    """
    if mode == "random":
        mode = np.random.choice(["pitch", "noise", "stretch"])

    if mode == "pitch":
        steps = np.random.choice([-6, -5, 5, 6])  # large unnatural shift
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

    elif mode == "noise":
        noise = np.random.normal(0, 0.035, len(y))
        return np.clip(y + noise, -1.0, 1.0)

    elif mode == "stretch":
        rate = np.random.choice([0.6, 0.7, 1.5, 1.6])  # too slow or too fast
        return librosa.effects.time_stretch(y, rate=rate)

    return y


def extract_features_from_array(y: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    """Same as extract_features() but accepts a numpy array directly."""
    import tempfile, soundfile as sf
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        sf.write(tmp.name, y, sr)
        return extract_features(tmp.name, sr)


print("Augmentation functions ready.")

## Step 6: Build the Labeled Dataset

In [ ]:
import soundfile as sf

feature_rows = []
labels       = []

print("Extracting features from audio files...")
for i, path in enumerate(audio_files):
    # ── Good pronunciation (label = 1) ─────────────────────────────────────
    feats_good = extract_features(path)
    if feats_good is None:
        continue
    feature_rows.append(feats_good)
    labels.append(1)

    # ── Degraded pronunciation (label = 0) ─────────────────────────────────
    try:
        y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
        y_aug = augment_audio(y, sr=sr)
        feats_bad = extract_features_from_array(y_aug, sr)
        if feats_bad is not None:
            feature_rows.append(feats_bad)
            labels.append(0)
    except Exception:
        pass

    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1} / {len(audio_files)} files")

X = np.array(feature_rows)
y_labels = np.array(labels)

print(f"\nDataset shape: {X.shape}")
print(f"Good samples: {(y_labels == 1).sum()} | Degraded samples: {(y_labels == 0).sum()}")

## Step 7: Train/Test Split and Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train_sc.shape} | Test: {X_test_sc.shape}")

## Step 8: Train Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train_sc, y_train)

y_pred_rf = rf_model.predict(X_test_sc)
rf_acc    = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Accuracy: {rf_acc:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=["Degraded", "Good"]))

## Step 9: Train a Small PyTorch Neural Network

In [ ]:
class PronunciationNet(nn.Module):
    """3-layer MLP for pronunciation quality classification."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


def train_nn(X_tr, y_tr, X_val, y_val, epochs=30, lr=1e-3):
    input_dim = X_tr.shape[1]
    model_nn  = PronunciationNet(input_dim).to(device)
    optimizer = torch.optim.Adam(model_nn.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(device)
    ytr = torch.tensor(y_tr, dtype=torch.long).to(device)
    Xva = torch.tensor(X_val, dtype=torch.float32).to(device)
    yva = torch.tensor(y_val, dtype=torch.long).to(device)

    train_losses, val_accs = [], []

    for epoch in range(1, epochs + 1):
        model_nn.train()
        optimizer.zero_grad()
        logits = model_nn(Xtr)
        loss   = criterion(logits, ytr)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model_nn.eval()
        with torch.no_grad():
            val_preds = model_nn(Xva).argmax(dim=1)
            val_acc   = (val_preds == yva).float().mean().item()
        val_accs.append(val_acc)

        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

    return model_nn, train_losses, val_accs


nn_model, train_losses, val_accs = train_nn(
    X_train_sc, y_train, X_test_sc, y_test, epochs=30
)

## Step 10: Plot Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, color="steelblue")
axes[0].set_title("Neural Network — Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

axes[1].plot(val_accs, color="coral")
axes[1].set_title("Neural Network — Validation Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.savefig(MODELS_DIR / "training_curve.png", dpi=120)
plt.show()
print("Training curve saved.")

## Step 11: Save All Models

In [ ]:
# Random Forest + scaler (sklearn)
joblib.dump(rf_model, MODELS_DIR / "random_forest.joblib")
joblib.dump(scaler,   MODELS_DIR / "scaler.joblib")

# PyTorch neural network
torch.save(nn_model.state_dict(), MODELS_DIR / "pronunciation_net.pt")

# Save feature config so notebooks 2 & 3 know the input shape
config = {"n_mfcc": N_MFCC, "input_dim": X.shape[1], "sample_rate": SAMPLE_RATE}
import json
with open(MODELS_DIR / "feature_config.json", "w") as f:
    json.dump(config, f)

print(f"All models saved to: {MODELS_DIR.resolve()}")
for p in MODELS_DIR.iterdir():
    print(f"  {p.name}")

## Step 12: Feature Importance (Random Forest)
Which features matter most for pronunciation quality?

In [ ]:
feature_names = (
    [f"MFCC_mean_{i}" for i in range(N_MFCC)] +
    [f"MFCC_std_{i}"  for i in range(N_MFCC)] +
    ["pitch_mean", "pitch_std", "rms_mean", "rms_std", "centroid_mean", "centroid_std"]
)

importances = rf_model.feature_importances_
top_idx     = np.argsort(importances)[-20:]  # top 20 features

plt.figure(figsize=(10, 6))
plt.barh([feature_names[i] for i in top_idx], importances[top_idx], color="steelblue")
plt.title("Top 20 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(MODELS_DIR / "feature_importance.png", dpi=120)
plt.show()

---
## Done!
Saved artifacts:
- `models/random_forest.joblib` — Random Forest classifier
- `models/pronunciation_net.pt` — PyTorch neural network weights
- `models/scaler.joblib` — Feature normalizer
- `models/feature_config.json` — Feature configuration

Proceed to **Notebook 2** for evaluation.